In [4]:
import dash
from dash import dcc
from dash import html
import plotly.graph_objs as go
import pandas as pd

# definir los datos a mostrar
from sqlalchemy import create_engine

In [6]:
# Cargar datos desde el CSV
df = pd.read_csv('olympics.csv')

# Convertir la columna 'Year' a datetime (ajusta el formato si es necesario)
df['Year'] = pd.to_datetime(df['Year'], format='%Y')

# Asegurarse de que las columnas de medallas sean numéricas
medal_columns = ['Gold', 'Silver', 'Bronze']
for column in medal_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

# Agrupar datos por año y sumar las medallas
df_grouped = (
    df.groupby('Year')
    .agg({
        'Gold': 'sum',
        'Silver': 'sum',
        'Bronze': 'sum'
    })
    .reset_index()
)

# Configurar estilos de línea para cada tipo de medalla
line_styles = {
    'Gold': {'color': 'gold', 'width': 4},
    'Silver': {'color': 'silver', 'width': 3, 'dash': 'dash'},
    'Bronze': {'color': 'brown', 'width': 2, 'dash': 'dot'},
}

# Definir los gráficos para cada medalla
data_medals_by_year = []
for column in line_styles.keys():
    data_medals_by_year.append(
        go.Scatter(
            x=df_grouped['Year'],
            y=df_grouped[column],
            mode='lines',
            line=line_styles[column],
            name=column
        )
    )

# Configurar el dashboard
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']
app = dash.Dash(__name__, external_stylesheets=external_stylesheets, compress=False)
app.layout = html.Div(
    children=[
        html.H1(children='Medallas Olímpicas por Año'),
        dcc.Graph(
            id='medals_by_year',
            figure={
                'data': data_medals_by_year,
                'layout': go.Layout(
                    title='Medallas Olímpicas por Año',
                    xaxis={'title': 'Año'},
                    yaxis={'title': 'Número de Medallas'}
                )
            }
        )
    ]
)


In [8]:
# Ejecutar el servidor de Dash
if __name__ == '__main__':
    app.run_server(host='0.0.0.0', port=3000)

ConnectionError: HTTPConnectionPool(host='0.0.0.0', port=3000): Max retries exceeded with url: /_alive_30287605-6832-41e7-9bc0-baf761d95ed9 (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000002AC5532C890>: Failed to establish a new connection: [WinError 10049] La dirección solicitada no es válida en este contexto'))